In [1]:
import sqlite3
import pandas as pd
import numpy as np

conn=sqlite3.connect(r'C:\Users\KRISH JHA\OneDrive\Desktop\Sepsis\data_handling\sepsis.db')

cursor=conn.cursor()

print("Database is Connected")

Database is Connected


In [3]:
feature_table=pd.read_sql_query("""
select * from feature_table
""",conn)

In [5]:
patient_outcome=feature_table.groupby('Patient_id')['SepsisLabel'].max().reset_index()

patient_outcome.head(15)

,Patient_id,SepsisLabel
0,p000001,0.0
1,p000002,0.0
2,p000003,0.0
3,p000004,0.0
4,p000005,0.0
5,p000006,0.0
6,p000007,0.0
7,p000008,0.0
8,p000009,1.0
9,p000010,0.0


In [ ]:
patient_outcome['SepsisLabel'].value_counts()

#7.2.. % are Sepsis

SepsisLabel
0.0    37404
1.0     2932
Name: count, dtype: int64

In [ ]:
from sklearn.model_selection import train_test_split


train_patients,test_patients=train_test_split(
    patient_outcome,
    test_size=0.3,
    random_state=42,
    stratify=patient_outcome['SepsisLabel']
)

print("Train patients:")
print(train_patients.head())

print("\nTest patients:")
print(test_patients.head())



Train patients:
      Patient_id  SepsisLabel
14458    p014459          0.0
18416    p018417          0.0
14860    p014861          0.0
2085     p002086          0.0
9515     p009516          0.0

Test patients:
      Patient_id  SepsisLabel
8643     p008644          0.0
24878    p104543          0.0
3782     p003783          0.0
29302    p108967          0.0
27826    p107491          0.0


In [9]:
train_ids=train_patients['Patient_id']
test_ids=test_patients['Patient_id']

train_data=feature_table[feature_table['Patient_id'].isin(train_ids)]
test_data=feature_table[feature_table['Patient_id'].isin(test_ids)]

In [11]:
overlap = (
    set(train_data["Patient_id"])
    &
    set(test_data["Patient_id"])
)

print(overlap)

set()


In [12]:
print("TRAIN patient outcomes:")

print(
    train_patients["SepsisLabel"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTEST patient outcomes:")

print(
    test_patients["SepsisLabel"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

TRAIN patient outcomes:
SepsisLabel
0.0    92.73
1.0     7.27
Name: proportion, dtype: float64

TEST patient outcomes:
SepsisLabel
0.0    92.73
1.0     7.27
Name: proportion, dtype: float64


In [14]:
print("Total patients:")
print(feature_table["Patient_id"].nunique())

print("\nTrain patients:")
print(train_data["Patient_id"].nunique())

print("\nTest patients:")
print(test_data["Patient_id"].nunique())

print("\nTrain rows:")
print(len(train_data))

print("\nTest rows:")
print(len(test_data))

Total patients:
40336

Train patients:
28235

Test patients:
12101

Train rows:
1086436

Test rows:
465774


In [17]:
drop_columns = [
    "Patient_id",
    "SepsisLabel",
    "Dataset"
]

feature_columns = [
    col for col in train_data.columns
    if col not in drop_columns
]

X_train = train_data[feature_columns].copy()
y_train = train_data["SepsisLabel"].copy()

X_test = test_data[feature_columns].copy()
y_test = test_data["SepsisLabel"].copy()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train.head()
y_train.head()

X_train: (1086436, 108)
y_train: (1086436,)
X_test: (465774, 108)
y_test: (465774,)


0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: SepsisLabel, dtype: float64

In [18]:
print("Train NaN:", X_train.isna().sum().sum())
print("Test NaN:", X_test.isna().sum().sum())

print("Train infinity:", np.isinf(X_train).sum().sum())
print("Test infinity:", np.isinf(X_test).sum().sum())

Train NaN: 0
Test NaN: 0
Train infinity: 0
Test infinity: 0
